In [17]:
!pip install scikit-learn==1.5.2
!pip install scikeras


Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN)

This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
Grid Search/Random Search: Use grid search or random search to try different architectures.
Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
A common practice is to start with 1–2 hidden layers.


In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.callbacks import EarlyStopping

import pickle


In [19]:
from google.colab import files

uploaded = files.upload()

Saving Churn_Modelling.csv to Churn_Modelling (1).csv


In [20]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [21]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [22]:
#Encode categorical variables
label_encoder=LabelEncoder()
data['Gender']=label_encoder.fit_transform(data['Gender'])

In [23]:
#one hot encode 'Geogrpahy'

onehot_encoder_geo=OneHotEncoder(handle_unknown='ignore')

geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']])

geo_encoded_df=pd.DataFrame(geo_encoded.toarray(),columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
data = pd.concat(
    [data.drop('Geography', axis=1), geo_encoded_df],
    axis=1
)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [24]:
X=data.drop('Exited',axis=1)
y=data['Exited']

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Scale these features using StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
#Save the encoders and scaler
#Save the encoder
with open('label_encoder_gender.pkl','wb') as file:
  pickle.dump(label_encoder,file)
with open('scaler.pkl','wb') as file:
  pickle.dump(scaler,file)
with open('onehot_encoder_geo.pkl','wb') as file:
  pickle.dump(onehot_encoder_geo,file)

In [26]:
from tensorflow.keras.layers import Input, Dense
def create_model(neurons=32, layers=1):
    model = Sequential()

    model.add(Input(shape=(X_train_scaled.shape[1],)))

    for i in range(layers):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [27]:
model = KerasClassifier(
    model=create_model,
    layers=1,
    neurons=32,
    epochs=50,
    batch_size=10,
    verbose=0
)

In [28]:
param_grid = {
    'model__neurons': [8, 16, 32, 64],
    'model__layers': [1, 2],
    'epochs': [10, 50, 100]
}

In [ ]:
#perform the grid search
grid=GridSearchCV(estimator=model,param_grid=param_grid,cv=3)
grid_result=grid.fit(X_train_scaled,y_train)
# print the best params
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Best: 0.858125 using {'epochs': 10, 'model__layers': 2, 'model__neurons': 16}


In [20]:
import sklearn
import scikeras
import tensorflow as tf

print("sklearn:", sklearn.__version__)
print("scikeras:", scikeras.__version__)
print("tensorflow:", tf.__version__)

sklearn: 1.6.1
scikeras: 0.13.0
tensorflow: 2.20.0


In [26]:
print(type(model))

<class 'scikeras.wrappers.KerasClassifier'>


In [44]:
import keras

print("Keras:", keras.__version__)
print("SciKeras:", scikeras.__version__)
print("Sklearn:", sklearn.__version__)

Keras: 3.13.2
SciKeras: 0.13.0
Sklearn: 1.6.1
